In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
plt.rcParams['font.family']= 'NanumGothic' 
plt.rcParams['axes.unicode_minus']=False

In [ ]:
df = pd.read_csv("df_concat_BA1.csv")

In [ ]:
df

In [ ]:
df_info = df.info()
df_head = df.head()
df_description = df.describe(include='all')# 데이터 기본 정보
info = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'unique': df.nunique()
}).sort_values(by='missing', ascending=False)
# to_string()으로 한 번에 출력
print(info.to_string())


In [ ]:
# df_description.to_string

In [ ]:
info[info['unique'] == 0]  #71840 rows × 257 columns

In [ ]:
# # 1. Null 값 개수 확인
# null_counts = df.isnull().sum()

# # 2. 변수 고유값 수 확인
# unique_counts = df.nunique()

# # 3. 데이터 타입 분류
# # 범주형 변수 기준: int 형이며 고유값 수가 30개 이하
# categorical_columns = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and unique_counts[col] <= 30]
# categorical_columns = list(set(categorical_columns) - {'기부여부'})  # 종속변수 제외
# numerical_columns = list(set(df.columns) - set(categorical_columns) - {'기부여부'})

# # 요약 정보 확인
# import pandas as pd
# summary_df = pd.DataFrame({
#     '결측치_개수': null_counts,
#     '고유값_수': unique_counts,
#     '데이터타입': df.dtypes
# })

# # 카테고리형 변수 및 수치형 변수 정보 출력
# print(summary_df)
# categorical_columns[:]

In [ ]:
## 주형 코드(int/float)이며 일부는 더미 변수 형태

In [ ]:
## 결측치 확인 후 40퍼 이상은 날리기 -- ver1
# 2) 컬럼별 결측 비율 계산
missing_ratio = df.isnull().mean()  # 각 컬럼의 결측치 비율
# 3) 결측 비율이 40% 미만인 컬럼만 남기기
cols_to_keep = missing_ratio[missing_ratio < 0.4].index
df_ver1 = df[cols_to_keep].copy()

# (선택) 삭제된 컬럼 확인
dropped_cols = missing_ratio[missing_ratio >= 0.4].index.tolist()
print(f"삭제된 컬럼 ({len(dropped_cols)}개):", dropped_cols)

# 결과 확인
print("원본 컬럼 수:", df.shape[1])
print("남은 컬럼 수:", df_ver1.shape[1])

In [ ]:
df_ver1.columns.tolist()

In [ ]:
# rows_with_missing_values = df_ver1[df_ver1.isna().any(axis=1)]
# rows_with_missing_values

In [ ]:
rows_missing_50 = df_ver1[df_ver1.isnull().mean(axis=1) >= 0.5]
rows_missing_50

In [ ]:
rows_missing_40 = df_ver1[df_ver1.isnull().mean(axis=1) >= 0.4]
rows_missing_40

In [ ]:
rows_missing_30 = df_ver1[df_ver1.isnull().mean(axis=1) >= 0.3]
rows_missing_30

In [ ]:
## 행기준 모두 결측치가 30퍼 미만이라 삭제하지 않음

In [ ]:
# 결측치 채우기
from sklearn.impute import SimpleImputer

def classify_and_impute(df, info):
    # 컬럼별 전략 분류
    strategy = {}
    
    for col in info.index:
        dtype = info.loc[col, 'dtype']
        missing = info.loc[col, 'missing']
        unique = info.loc[col, 'unique']
        
        # 결측치가 없으면 건너뜀
        if missing == 0:
            continue
        
        # 1. 고유값이 2~6개 정도의 범주형 --> 최빈값
        if unique <= 6:
            strategy[col] = 'most_frequent'
        
        # 2. 고유값이 많고 연속형 수치 같으면 --> 평균 or 중앙값
        elif unique >= 30:
            strategy[col] = 'median'
        
        # 3. 중간값 (10~30개) 정도면 --> 중앙값 
        else:
            strategy[col] = 'median'
    
    # 실제 채우기 적용
    for col, method in strategy.items():
        if method == 'most_frequent':
            df[col] = df[col].fillna(df[col].mode()[0])
        elif method == 'median':
            df[col] = df[col].fillna(df[col].median())
        elif method == 'mean':
            df[col] = df[col].fillna(df[col].mean())
    
    return df, strategy

# 결과 출력을 위한 구조 반환
def summarize_strategy(info, strategy):
    summary = []
    for col, method in strategy.items():
        summary.append({
            'column': col,
            'dtype': info.loc[col, 'dtype'],
            'missing': info.loc[col, 'missing'],
            'unique': info.loc[col, 'unique'],
            'strategy': method
        })
    return pd.DataFrame(summary)



# info 추출
info = pd.DataFrame({
    'dtype': df_ver1.dtypes,
    'missing': df_ver1.isnull().sum(),
    'unique': df_ver1.nunique()
})

# 결측치 채우기 실행
df_filled, strategy_used = classify_and_impute(df_ver1.copy(), info)

# 전략 요약표 생성
summary = summarize_strategy(info, strategy_used)

print(summary)


In [ ]:
# 각 컬럼별 결측치 수
print(df_filled.isnull().sum())

# 전체 결측치 수
print(f"전체 결측치 수: {df_filled.isnull().sum().sum()}")

# RobustScaler 적용

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. X, y 분리 (기부여부는 종속변수)
X = df_filled.drop(columns='기부여부')
y = df_filled['기부여부']

# 2. 데이터 타입 일괄 변환 (숫자로)
X = X.apply(pd.to_numeric, errors='coerce')  # 혹시 남은 문자형 대비
X = X.fillna(0)  # 변환 실패로 생긴 NaN 다시 처리

# 3. 학습/테스트 데이터 분리 (8:2 비율, stratify로 클래스 균형 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. RobustScaler 적용
scaler = RobustScaler()
scaler.fit(X_train)  # 학습 데이터로만 fit

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. 결과 확인
result = {
    "X_train shape": X_train_scaled.shape,
    "y_train shape": y_train.shape,
    "X_train min": X_train_scaled.min(),
    "X_train max": X_train_scaled.max(),
    "X_test shape": X_test_scaled.shape,
    "y_test shape": y_test.shape,
    "X_test min": X_test_scaled.min(),
    "X_test max": X_test_scaled.max(),
    "status": "Complete!"
}

result

In [ ]:
##운에 따라 테스트셋에 기부자가 너무 적거나 많아질 수 있어서 stratify=y를 쓰면 학습셋과 테스트셋 둘 다 80:20 비율 유지되도록 만듭니다.

# 모델링 학습 : Logistic Regression, Random Forest, XGBoost, LightGBM, CatBoost, MLP, CNN 7종

In [ ]:
# 1. 필수 라이브러리 임포트
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Dropout

import pandas as pd
import numpy as np

# 2. 모델 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
}

# 3. 학습 및 평가
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    report = classification_report(y_test, y_pred, output_dict=True)
    results[name] = report['weighted avg']

# 4. CNN 모델 학습
print("Training CNN...")
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_cnn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

cnn_model = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_scaled.shape[1], 1)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

cnn_model.fit(X_train_cnn, y_train, epochs=5, batch_size=128, verbose=1)

y_pred_cnn = (cnn_model.predict(X_test_cnn) > 0.5).astype("int32")
cnn_report = classification_report(y_test, y_pred_cnn, output_dict=True)
results['CNN'] = cnn_report['weighted avg']

# 5. 결과 출력
results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

# 결과 저장용 리스트
summary = []

# 기존 모델들 평가 반복
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else y_pred

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    auc = roc_auc_score(y_test, y_proba)

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * precision * recall / (precision + recall)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    balanced_acc = (recall + specificity) / 2

    summary.append({
        'Dataset': 'Test',
        'Algorithm': name,
        'N': len(y_test),
        'True Positive': tp,
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1-score': f1,
        'Accuracy': accuracy,
        'Balanced Accuracy': balanced_acc,
        'AUC': auc
    })

# CNN 따로 평가
y_pred_cnn = (cnn_model.predict(X_test_cnn) > 0.5).astype('int32')
y_proba_cnn = cnn_model.predict(X_test_cnn).ravel()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cnn).ravel()
auc = roc_auc_score(y_test, y_proba_cnn)

precision = tp / (tp + fp)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = 2 * precision * recall / (precision + recall)
accuracy = (tp + tn) / (tp + tn + fp + fn)
balanced_acc = (recall + specificity) / 2

summary.append({
    'Dataset': 'Test',
    'Algorithm': 'CNN',
    'N': len(y_test),
    'True Positive': tp,
    'True Negative': tn,
    'False Positive': fp,
    'False Negative': fn,
    'Precision': precision,
    'Recall': recall,
    'Specificity': specificity,
    'F1-score': f1,
    'Accuracy': accuracy,
    'Balanced Accuracy': balanced_acc,
    'AUC': auc
})

# 결과를 DataFrame으로 정리 및 정렬
results_df = pd.DataFrame(summary)
results_df = results_df.sort_values(by='AUC', ascending=False).reset_index(drop=True)

# 출력
results_df


In [ ]:
## weighted 성능 평가 방식

In [ ]:
## 파라미터 조정 후 
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
#from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# 1. LightGBM 하이퍼파라미터 튜닝 (class_weight 적용)
lgb_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [4, 6],
    'learning_rate': [0.01, 0.1],
    'num_leaves': [20, 31]
}
lgb_grid = GridSearchCV(
    lgb.LGBMClassifier(class_weight='balanced', random_state=42), 
    lgb_param_grid, scoring='f1', cv=3
)
lgb_grid.fit(X_train_scaled, y_train)
best_lgb = lgb_grid.best_estimator_

# 2. CatBoost 하이퍼파라미터 튜닝 (class_weights 직접 지정 필요)
# 자동 계산: [weight_0, weight_1]
w_0 = (y_train == 0).sum()
w_1 = (y_train == 1).sum()
class_weights = [1, w_0 / w_1]

cat_param_grid = {
    'iterations': [100, 300],
    'depth': [4, 6],
    'learning_rate': [0.01, 0.1]
}
cat_grid = GridSearchCV(
    CatBoostClassifier(class_weights=class_weights, verbose=0, random_state=42),
    cat_param_grid, scoring='f1', cv=3
)
cat_grid.fit(X_train_scaled, y_train)
best_cat = cat_grid.best_estimator_

# 3. 임계값 조정 (LightGBM 기준)
y_proba = best_lgb.predict_proba(X_test_scaled)[:, 1]
thresholds = [0.3, 0.4, 0.5]
threshold_tuning = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    acc = (tp + tn) / (tp + tn + fp + fn)
    balanced_acc = (recall + (tn / (tn + fp))) / 2
    auc = roc_auc_score(y_test, y_proba)

    threshold_tuning.append({
        "Threshold": t,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": balanced_acc,
        "AUC": auc
    })

threshold_df = pd.DataFrame(threshold_tuning)

# 4. 앙상블 (LightGBM + CatBoost + LogisticRegression)
ensemble_model = VotingClassifier(
    estimators=[
        ('lgb', best_lgb),
        ('cat', best_cat),
        ('log', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ],
    voting='soft'
)
ensemble_model.fit(X_train_scaled, y_train)
ensemble_pred = ensemble_model.predict(X_test_scaled)
ensemble_report = classification_report(y_test, ensemble_pred, output_dict=True)

# 결과 출력
threshold_df


In [ ]:
ensemble_report

| 목적                        | 임계값  | 이유                              |
| ------------------------- | ------- | ------------------------------- |
| **기부자 최대 확보 (Recall 우선)** | **0.3** | Recall 0.90 → 기부자를 가장 많이 포착함    |
| **균형 잡힌 성능 (F1 최대)**      | **0.5** | F1-score 가장 높고, Accuracy도 가장 좋음 |
| **보수적 예측 (Precision 우선)** | 0.5 이상  | Precision 0.56로 오답 줄이기 유리       |


In [ ]:
#  weighted 성능 평가 방식으로 모델 성능 비교


# 모델 정의 (class_weight='balanced' 적용 가능 모델에만 설정)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, scale_pos_weight=3.5),
    "LightGBM": lgb.LGBMClassifier(class_weight='balanced', random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
}

# 결과 저장
weighted_results = {}

# 모델별 학습 및 weighted 성능 추출
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    report = classification_report(y_test, y_pred, output_dict=True)
    weighted_results[name] = report['weighted avg']

# DataFrame으로 정리
results_df = pd.DataFrame(weighted_results).T
results_df


In [ ]:
#!pip install -U scikit-learn

In [ ]:
# 결과 저장용 리스트
full_summary = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else y_pred

    # weighted 성능
    report = classification_report(y_test, y_pred, output_dict=True)
    weighted = report['weighted avg']

    # confusion matrix 기반 성능
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * (precision * recall) / (precision + recall)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    balanced_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_test, y_proba)

    full_summary.append({
        'Algorithm': name,
        'N': len(y_test),
        'True Positive': tp,
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1-score': f1,
        'Accuracy': accuracy,
        'Balanced Accuracy': balanced_acc,
        'AUC': auc,
        'Weighted Precision': weighted['precision'],
        'Weighted Recall': weighted['recall'],
        'Weighted F1': weighted['f1-score']
    })

# DataFrame으로 정리
full_results_df = pd.DataFrame(full_summary)
full_results_df

# 아예 전처리 간단하게 해서 성능 보기

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten
from tensorflow.keras.utils import to_categorical

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'  # 종속변수 이름을 실제 값으로 바꿔주세요

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 2. 모델링 및 평가
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 3. CNN 모델
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 4. 결과 DataFrame 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'  
# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#  StandardScaler 사용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 2. 모델링 및 평가
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 3. CNN 모델
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 4. 결과 DataFrame 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'  # ← 실제 타겟 컬럼명으로 바꾸세요

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링 없음

# 결과 저장 리스트
results = []

# 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 2. 모델 학습 및 평가
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 3. CNN 모델
# CNN 입력 차원 맞추기
X_train_cnn = X_train.to_numpy().reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.to_numpy().reshape(X_test.shape[0], X_test.shape[1], 1)

cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 4. 결과 정리
result_df = pd.DataFrame(results)
result_df


In [ ]:
# Undersampling
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import OneSidedSelection

In [ ]:
def undersampling(X_train, Y_train, method='OneSidedSelection', strategy='auto', random_state=123):
    print('Ratio of Origin Y_train: ', Y_train.value_counts())
    if method == 'RandomUnderSampler':
        sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
    elif method == 'TomekLinks':
        sampler = TomekLinks(sampling_strategy=strategy)
    elif method == 'CondensedNearestNeighbour':
        sampler = CondensedNearestNeighbour(sampling_strategy=strategy, random_state=random_state)
    elif method == 'OneSidedSelection':
        sampler = OneSidedSelection(sampling_strategy=strategy, random_state=random_state)
        
    # 축소
    X_train, Y_train = sampler.fit_resample(X_train, Y_train)
    print('Ratio of Sapling Y_train: ', Y_train.value_counts())
    
    return sampler, X_train, Y_train


undersampling(X_train, y_train)

In [ ]:
# ['가구일련번호', '가구원번호', '가구주관계코드', '가구가중값', '가구원가중값'] 

# 다시

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.under_sampling import (
    RandomUnderSampler,
    TomekLinks,
    CondensedNearestNeighbour,
    OneSidedSelection
)

# 1. 데이터 불러오기
df = pd.read_csv("df_concat_BA1.csv")

# 2. 결측치 처리 함수
def custom_missing_handler(df):
    total_rows = len(df)

    # 고결측 컬럼 삭제
    high_missing_cols = df.columns[df.isnull().sum() / total_rows > 0.6]
    df = df.drop(columns=high_missing_cols, errors='ignore')
    print(f"삭제한 고결측 컬럼 수: {len(high_missing_cols)}")

    # 연속형 변수 → 0 대체
    continous_patterns = ['시간수', '횟수', '권수', '일수']
    continous_cols = [col for col in df.columns if any(p in col for p in continous_patterns)]
    df[continous_cols] = df[continous_cols].fillna(0)
    print(f"연속형 컬럼 수: {len(continous_cols)} → 결측 0 대체")

    # Yes/No 변수 → 0 대체
    yn_patterns = ['여부']
    yn_cols = [col for col in df.columns if any(p in col for p in yn_patterns)]
    df[yn_cols] = df[yn_cols].fillna(0)
    print(f"Yes/No 컬럼 수: {len(yn_cols)} → 결측 0 대체")

    # 코드형 변수 → -1 대체
    code_cols = [col for col in df.columns if col.endswith('코드')]
    df[code_cols] = df[code_cols].fillna(-1)
    print(f"카테고리 코드 컬럼 수: {len(code_cols)} → 결측 -1 대체")

    # 남은 결측 처리
    remain_cols = df.columns[df.isnull().any()]
    for col in remain_cols:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna('미응답')
    print(f"남은 결측 처리: {len(remain_cols)} 개")
    print(f"최종 남은 결측치 수: {df.isnull().sum().sum()}")

    return df

# 3. 언더샘플링 함수
def undersampling(X_train, Y_train, method='OneSidedSelection', strategy='auto', random_state=123):
    print('Ratio of Origin Y_train: \n', Y_train.value_counts())

    if method == 'RandomUnderSampler':
        sampler = RandomUnderSampler(sampling_strategy=strategy, random_state=random_state)
    elif method == 'TomekLinks':
        sampler = TomekLinks(sampling_strategy=strategy)
    elif method == 'CondensedNearestNeighbour':
        sampler = CondensedNearestNeighbour(sampling_strategy=strategy, random_state=random_state)
    elif method == 'OneSidedSelection':
        sampler = OneSidedSelection(sampling_strategy=strategy, random_state=random_state)

    X_resampled, Y_resampled = sampler.fit_resample(X_train, Y_train)
    print('Ratio of Sampled Y_train: \n', Y_resampled.value_counts())

    return sampler, X_resampled, Y_resampled

# 4. 전체 전처리 + 학습/테스트 분리 + 스케일링 + 언더샘플링
def preprocessing_pipeline(df, target_col='기부여부'):
    df_clean = custom_missing_handler(df.copy())

    # X, y 분리
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]

    # 학습/테스트 분리
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # MinMaxScaler 적용
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 언더샘플링 (학습 데이터에만)
    sampler, X_train_resampled, y_train_resampled = undersampling(
        X_train_scaled, y_train, method='OneSidedSelection'
    )

    return X_train_resampled, X_test_scaled, y_train_resampled, y_test, scaler, sampler

# 5. 실행
X_train, X_test, y_train, y_test, scaler, sampler = preprocessing_pipeline(df, target_col='기부여부')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score, roc_auc_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten

# 1. 성능 저장 리스트
results = []

# 2. 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 3. 머신러닝 모델링 및 평가
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 4. CNN 모델용 reshape
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 5. CNN 모델 구성 및 평가
cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 6. 결과 DataFrame 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
from imblearn.over_sampling import SMOTE

# 1. 데이터 불러오기
df = pd.read_csv("df_concat_BA1.csv")

# 2. 결측치 처리 함수
def custom_missing_handler(df):
    total_rows = len(df)

    # 고결측 컬럼 삭제
    high_missing_cols = df.columns[df.isnull().sum() / total_rows > 0.6]
    df = df.drop(columns=high_missing_cols, errors='ignore')

    # 연속형 변수 → 0 대체
    continous_patterns = ['시간수', '횟수', '권수', '일수']
    continous_cols = [col for col in df.columns if any(p in col for p in continous_patterns)]
    df[continous_cols] = df[continous_cols].fillna(0)

    # Yes/No 변수 → 0 대체
    yn_patterns = ['여부']
    yn_cols = [col for col in df.columns if any(p in col for p in yn_patterns)]
    df[yn_cols] = df[yn_cols].fillna(0)

    # 코드형 변수 → -1 대체
    code_cols = [col for col in df.columns if col.endswith('코드')]
    df[code_cols] = df[code_cols].fillna(-1)

    # 남은 결측 처리
    remain_cols = df.columns[df.isnull().any()]
    for col in remain_cols:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna('미응답')

    return df

# 3. 전체 전처리 + 학습/테스트 분리 + 스케일링 + 오버샘플링(SMOTE)
def preprocessing_pipeline(df, target_col='기부여부'):
    df_clean = custom_missing_handler(df.copy())

    # X, y 분리
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]

    # 학습/테스트 분리
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # MinMaxScaler 적용
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 오버샘플링 (학습 데이터에만)
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    return X_train_resampled, X_test_scaled, y_train_resampled, y_test, scaler, smote

# 4. 실행
X_train, X_test, y_train, y_test, scaler, sampler = preprocessing_pipeline(df, target_col='기부여부')

# 5. 성능 저장 리스트
results = []

# 6. 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 7. 머신러닝 모델링 및 평가
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced'),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(class_weight='balanced'),
    "CatBoost": CatBoostClassifier(verbose=0, class_weights=[1, 3]),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 8. CNN 모델용 reshape
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 9. CNN 모델 구성 및 평가
cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 10. 결과 DataFrame 출력
result_df = pd.DataFrame(results)
result_df 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten
from imblearn.under_sampling import RandomUnderSampler

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler 사용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 언더샘플링 적용 (train 데이터에만)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train_scaled, y_train)

# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 모델 정의 및 평가 (undersampled 데이터로 학습)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# CNN 모델용 reshape
X_train_cnn = X_train_res.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# CNN 모델 학습
cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train_res, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 언더샘플링 + XGBoost 단독 평가 출력 (원 요청 코드 포함)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_res, y_train_res)
y_pred_final = xgb_model.predict(X_test_scaled)

conf_matrix = confusion_matrix(y_test, y_pred_final)
report = classification_report(y_test, y_pred_final)
acc = accuracy_score(y_test, y_pred_final)
f1_weighted = f1_score(y_test, y_pred_final, average='weighted')
f1_original = f1_score(y_test, y_pred_final)

# 성능 요약표 출력
result_df = pd.DataFrame(results)
result_df
# 추가 리포트도 함께 반환
(conf_matrix, report, acc, f1_weighted, f1_original)


In [ ]:
# 모델을 XGBoost만 사용하도록 수정한 코드 실행
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, classification_report
)
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten
from imblearn.under_sampling import RandomUnderSampler

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler 사용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 언더샘플링 적용 (train 데이터에만)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train_scaled, y_train)

# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# XGBoost 모델 학습 및 평가
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

# CNN 모델용 reshape
X_train_cnn = X_train_res.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# CNN 모델 학습
cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train_res, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 성능 요약표 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
# XGBoost만 사용 + MinMaxScaler 적용 + 언더샘플링 제거한 버전
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from xgboost import XGBClassifier

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MinMaxScaler 사용
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# XGBoost 모델 학습 및 평가
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

# 성능 요약표 출력
result_df = pd.DataFrame(results)
result_df 

In [ ]:
# XGBoost만 사용 + MinMaxScaler 적용
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from xgboost import XGBClassifier

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MinMaxScaler 사용
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 언더샘플링 적용 (train 데이터에만)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train_scaled, y_train)


# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# XGBoost 모델 학습 및 평가 (언더샘플링된 데이터 사용)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_res, y_train_res)  # ← 여기 수정됨!
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

# 성능 요약표 출력
result_df = pd.DataFrame(results)
result_df 

In [ ]:
# 수정된 전체 코드: XGBoost + MinMaxScaler + 언더샘플링 반영 + 하이퍼파라미터 간소화 버전
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 최빈값으로 결측 채우기
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# X, y 분리
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# train-test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MinMaxScaler 사용
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 언더샘플링 적용 (train 데이터에만)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train_scaled, y_train)

# 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# XGBoost 모델 정의 (간소화된 하이퍼파라미터 설정)
xgb_model = XGBClassifier(
    max_depth=4,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# 학습 및 평가
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

# 성능 요약표 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
# 7개 모델(XGBoost 포함) 비교용 코드: 공통된 간소화된 설정 반영
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten
import numpy as np

# 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 성능 저장 리스트
results = []

# 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 모델 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost": XGBClassifier(max_depth=4, learning_rate=0.1, n_estimators=200, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, use_label_encoder=False,
                             eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(max_depth=4, learning_rate=0.1, n_estimators=200, subsample=0.8,
                               colsample_bytree=0.8, min_child_samples=10, random_state=42),
    "CatBoost": CatBoostClassifier(depth=4, learning_rate=0.1, iterations=200, verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, learning_rate_init=0.01, random_state=42)
}

# 학습 및 평가
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# CNN 별도 처리
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 결과 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
# 언더샘플링 + 7종 모델 비교 (튜닝된 하이퍼파라미터 반영) 코드
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten
from imblearn.under_sampling import RandomUnderSampler
import numpy as np

# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 언더샘플링 적용 (train 데이터에만)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train_scaled, y_train)

# 2. 성능 저장 리스트
results = []

# 공통 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 3. 모델 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost": XGBClassifier(max_depth=4, learning_rate=0.1, n_estimators=200, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, use_label_encoder=False,
                             eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(max_depth=4, learning_rate=0.1, n_estimators=200, subsample=0.8,
                               colsample_bytree=0.8, min_child_samples=10, random_state=42),
    "CatBoost": CatBoostClassifier(depth=4, learning_rate=0.1, iterations=200, verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, learning_rate_init=0.01, random_state=42)
}

# 4. 모델 학습 및 평가
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# 5. CNN 모델 별도 처리
X_train_cnn = X_train_res.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train_res, epochs=10, batch_size=32, verbose=0)

y_proba_cnn = cnn.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 결과 출력
result_df = pd.DataFrame(results)
result_df


In [ ]:
# 하이퍼파라미터 튜닝 반영한 7종 모델 비교 코드 (LightGBM 제외)
from sklearn.utils.class_weight import compute_class_weight

# 성능 저장 리스트 초기화
results = []

# 공통 평가 함수 재사용
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    results.append({
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    })

# 튜닝된 모델 정의
tuned_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_split=5, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(
        max_depth=5, learning_rate=0.05, n_estimators=400, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=5, use_label_encoder=False,
        eval_metric='logloss', scale_pos_weight=2.5, random_state=42),
    "CatBoost": CatBoostClassifier(
        depth=5, learning_rate=0.05, iterations=400, random_seed=42, verbose=0),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=500, learning_rate_init=0.005, random_state=42)
}

# 학습 및 평가 (LightGBM 제외)
for name, model in tuned_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_proba)

# CNN 튜닝
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn_tuned = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn_tuned.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_tuned.fit(X_train_cnn, y_train, epochs=20, batch_size=32, verbose=0)

y_proba_cnn = cnn_tuned.predict(X_test_cnn).flatten()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
evaluate_model("CNN", y_test, y_pred_cnn, y_proba_cnn)

# 결과 출력
result_df_tuned = pd.DataFrame(results)
import ace_tools as tools; tools.display_dataframe_to_user(name="튜닝된 7종 모델 성능 비교 결과", dataframe=result_df_tuned)


In [ ]:
# 데이터 로드
df = pd.read_csv("df_concat_BA1.csv")

# 1) 각 행의 결측 비율 계산 (axis=1 → 행 단위)
row_missing_ratio = df.isnull().mean(axis=1)

# 2) 결측 비율 ≥ 0.5(50%)인 행만 추출
df_high_missing = df[row_missing_ratio >= 0.4].copy()

# (선택) 몇 개의 행이 해당되는지 확인
print(f"결측 비율 50% 이상인 행 개수: {df_high_missing.shape[0]} / {df.shape[0]}")

# 3) 실제 행 데이터 출력 (원하면 .head() 등으로 일부만 보기)
print(df_high_missing)

In [ ]:
rows_with_missing_values = df[df.isna().any(axis=1)]
print("결측치가 있는 행:")
print(rows_with_missing_values)